In [2]:
## ======= ##
## IMPORTS ##
## ======= ##

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import zarr
import rioxarray
import matplotlib.pyplot as plt
import os
import dask
import dask.array
import math

import datetime

from collections import Counter

import pystac_client
from pystac.extensions.projection import ProjectionExtension as proj

import planetary_computer
import rasterio
import rasterio.features
from rasterio.features import rasterize

import stackstac
import pyproj

import dask.diagnostics

from shapely.geometry import box
from shapely.ops import transform

from scipy.ndimage import binary_propagation
from scipy.ndimage import label

import sat_tile_stack
from sat_tile_stack import sattile_stack, sat_mask_array, write_netcdf_from_da
from sat_tile_stack import *



In [12]:
# scratch/joshua/lakesproj/tstacks/SW2019_tstacks
ds_tstack = xr.open_dataset("/home/jupyter/data/SW2019_tstacks/tstack_SW2019_0.nc")
ds_tstack




<xarray.Dataset> Size: 1GB
Dimensions:              (x: 512, y: 512, time: 153, band: 8)
Coordinates: (12/15)
  * x                    (x) float64 4kB 5.595e+05 5.595e+05 ... 5.646e+05
  * y                    (y) float64 4kB 7.538e+06 7.538e+06 ... 7.533e+06
  * time                 (time) datetime64[ns] 1kB 2019-05-01 ... 2019-09-30
    constellation        <U10 40B ...
    instruments          <U3 12B ...
    title                (band) <U26 832B ...
    ...                   ...
    full_width_half_max  (band) float64 64B ...
    epsg                 int64 8B ...
    eo_cloud_cover       (time) float64 1kB ...
    pct_nans             (time) float64 1kB ...
    pctnanpix_inmask     (time) float64 1kB ...
    pctcloudypix_inmask  (time) float64 1kB ...
Dimensions without coordinates: band
Data variables:
    reflectance          (time, band, y, x) float32 1GB ...
Attributes:
    crs:                                      epsg:32622
    transform:                                [ 1.00000e+01  0.00000e+00  5.5...
    resolution:                               10
    band:                                     ['B04', 'B03', 'B02', 'B08', 'B...
    s2:mgrs_tile:                             22WEA
    s2:degraded_msi_data_percentage:          0.0
    s2:saturated_defective_pixel_percentage:  0.0
    s2:processing_baseline:                   02.12
    s2:datatake_type:                         INS-NOBS
    s2:product_type:                          S2MSI2A
    proj:code:                                EPSG:32622
    proj:bbox:                                [ 609780. 7490220. 7600020.  49...

In [13]:
# nans:  (note that this will place a 1 in the time vector if there is 100% NaNs within the lake area)
pct_nan = pctnanpix_inmask(ds_tstack)
ds_tstack = ds_tstack.assign_coords(pctnanpix_inmask=("time", pct_nan.data))
# cloudiness: (note that this will place a 1 in the time vector if there is 100% cloudiness within the lake area)
# NOTE: this currently also places a 1 in the time vector if the image has NaNs inside the lake area
pct_cloudy = pctcloudypix_inmask(ds_tstack)
ds_tstack = ds_tstack.assign_coords(pctcloudypix_inmask=("time", pct_cloudy.data))


KeyError: 'band'